In [7]:
# Load environment variables and verify the project setup.
import sys
from pathlib import Path

# Find the repo root (the folder containing env_checker.py) and make it importable.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "env_checker.py").exists())
sys.path.insert(0, str(ROOT))

# Load .env into the environment for this session.
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ModuleNotFoundError:
    print("python-dotenv not installed yet — run: uv add python-dotenv")

# Verify .env variables and required packages.
from env_checker import run_checks
run_checks()

Environment variables (from .env.example)
  ✓ OPENAI_API_KEY  — set
  ✓ ANTHROPIC_API_KEY  — set
  ✓ LANGSMITH_TRACING  — set
  ✓ LANGSMITH_ENDPOINT  — set
  ✓ LANGSMITH_API_KEY  — set
  ✓ LANGSMITH_PROJECT  — set
  ✓ CHROMA_PERSIST_DIR  — set

Required packages (from pyproject.toml)
  ✓ beautifulsoup4  — installed (4.14.3)
  ✓ chromadb  — installed (1.5.9)
  ✓ langchain  — installed (1.3.2)
  ✓ langchain-chroma  — installed (1.1.0)
  ✓ langchain-community  — installed (0.4.2)
  ✓ langchain-core  — installed (1.4.0)
  ✓ langchain-openai  — installed (1.2.2)
  ✓ lxml  — installed (6.1.1)
  ✓ onnxruntime  — installed (1.19.2)
  ✓ pypdf  — installed (6.12.2)
  ✓ python-dotenv  — installed (1.2.2)
  ✓ ipykernel  — installed (7.2.0)
  ✓ jupyterlab  — installed (4.5.7)

✓ All checks passed.


True

# LCEL (LangChain Expression Language)

Compose retrieval + prompt + model + parser into a single chain with the `|` pipe operator (e.g. `retriever | prompt | llm | parser`). Supports streaming, batching, and async out of the box.

_TODO: build a retrieval chain over the `rag_reference` Chroma collection._

# Examples(PDF)


### 1.Load the documents

In [8]:
from langchain_community.document_loaders import PyPDFLoader

file_path = ROOT / "assets/sample-docs/sdlc-end-to-end.pdf"

loader = PyPDFLoader(str(file_path))
documents = loader.load()
print(f"Loaded {len(documents)} page(s)")

Loaded 12 page(s)


### 2. Chuncking(Recursive Chunking)

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators= ["\n\n", "\n", " ", ""]
)

spliting the documents

In [10]:
doc_chunks = text_splitter.split_documents(documents)


### 3.Embeddings

In [11]:
from langchain_openai import OpenAIEmbeddings

openai_embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

### 4. Vector store

In [12]:
import os
from langchain_chroma import Chroma

# Anchor the DB to the repo root so it lands in the same place from any notebook.
persist_dir = str(ROOT / os.getenv("CHROMA_PERSIST_DIR", "./chroma_db"))

vector_store = Chroma(
    collection_name="rag_reference",
    embedding_function=openai_embedding_model,
    persist_directory=persist_dir,
)
print("Collection:", vector_store._collection.name)
print("Persist dir:", persist_dir)

Collection: rag_reference
Persist dir: /Users/Prabhukumar/Projects/PycharmProjects/rag-reference/chroma_db


createing embeddings and storing in vector_store

In [13]:
# Embed the chunks and write them into the persistent collection.
# (Re-running adds duplicates; reset the collection first if you re-ingest.)
ids = vector_store.add_documents(doc_chunks)

print(f"Added {len(ids)} chunks to collection '{vector_store._collection.name}'")
print("Total in collection:", vector_store._collection.count())

Added 29 chunks to collection 'rag_reference'
Total in collection: 87


### 1. Configure Model

In [14]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(model="gpt-5-nano")

### 2. Creating prompt tempalate

In [15]:
from langchain_core.prompts import ChatPromptTemplate

prompt = """
Use the only the context provided to answer the following question. If you don't know the answer, reply that you are unsure.
Context: {context}
Question: {question}
"""

# Convert the string into a chat prompt template
prompt_template = ChatPromptTemplate.from_template(prompt)


### Retriever

In [16]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k":2}
)

### Creating the chain

In [17]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chain = (
    {"context": retriever,"question":RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

## Final testing

In [22]:
response = chain.invoke("What is LLD")

In [23]:
print(response)

LLD stands for Low-Level Design. It provides the detailed internal design of each component—the blueprint developers code from. It includes class/module structure, methods and responsibilities, and detailed algorithms, data structures, and pseudo-code.
